# 〇×ゲーム ニューラルネットワーク実験（Colab用）

上のセルから順に実行してください。**編集は不要です。**

## 手順1. プログラムを読み込む
- `/content` に `tictactoe-nn-lab*.zip` があれば、それを使います（Colab左のファイル欄にドラッグ&ドロップしておくと確実です）。
- ZIPがなければ、GitHubのリポジトリから取得します（リポジトリが公開されている場合）。
- どちらもできなければ、ZIPのアップロード画面が出ます。

In [ ]:
SOURCE = "auto"   # "auto" = ZIPがあればZIP、なければGitHub、それも無理ならアップロード / "zip" / "github"
GITHUB_URL = "https://github.com/pale76/tictactoe-nn-lab.git"

import glob, os, shutil, subprocess, sys, zipfile

TARGET = "/content/tictactoe-nn-lab"


def find_zips():
    return sorted(glob.glob("/content/tictactoe-nn-lab*.zip"))


def use_zip(zips):
    newest = max(zips, key=os.path.getmtime)          # 複数あれば、いちばん新しいもの
    shutil.rmtree(TARGET, ignore_errors=True)         # 古いフォルダが残っていると混ざるため、先に消す
    zipfile.ZipFile(newest).extractall("/content")
    print("使用したZIP:", newest)


def upload_zip():
    from google.colab import files
    print("tictactoe-nn-lab.zip を選択してください")
    print("（ボタンが出ないときは、このセルをもう一度実行するか、左のファイル欄から /content にZIPを入れて再実行）")
    uploaded = files.upload()
    zips = [os.path.join("/content", n) for n in uploaded if n.lower().endswith(".zip")]
    if not zips:
        raise FileNotFoundError("ZIPが見つかりません。/content に tictactoe-nn-lab.zip を入れてください。")
    use_zip(zips)


def clone_github():
    """GitHubから取得する。成功したら True。"""
    shutil.rmtree(TARGET, ignore_errors=True)
    try:
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, TARGET], check=True, timeout=180,
                       env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})   # 認証を求められたら失敗にする
        print("GitHubから取得しました:", GITHUB_URL)
        return True
    except Exception as e:
        print("GitHubから取得できませんでした（リポジトリが未公開・非公開の可能性）:", type(e).__name__)
        shutil.rmtree(TARGET, ignore_errors=True)
        return False


if SOURCE == "zip":
    zips = find_zips()
    use_zip(zips) if zips else upload_zip()
elif SOURCE == "github":
    if not clone_github():
        raise RuntimeError("GitHubから取得できませんでした。GITHUB_URL とリポジトリの公開設定を確認してください。")
elif SOURCE == "auto":
    zips = find_zips()
    if zips:
        use_zip(zips)
    elif not clone_github():
        upload_zip()
else:
    raise ValueError('SOURCE は "auto" / "zip" / "github" のどれかにしてください')

hits = sorted(glob.glob("/content/**/src/tictactoe_nn/__init__.py", recursive=True))
if not hits:
    raise FileNotFoundError("src/tictactoe_nn が見つかりません。ZIPやリポジトリの中身を確認してください。")
SRC = os.path.dirname(os.path.dirname(hits[0]))   # .../src
ROOT = os.path.dirname(SRC)                       # リポジトリ直下
for name in [m for m in sys.modules if m == "tictactoe_nn" or m.startswith("tictactoe_nn.")]:
    del sys.modules[name]                         # 前に読み込んだ古いプログラムを忘れさせる
if SRC in sys.path:
    sys.path.remove(SRC)
sys.path.insert(0, SRC)
os.chdir(ROOT)
print("作業フォルダ:", ROOT)

## 手順2. ライブラリをインストール
ColabにはPyTorchなどが入っているので、主に Gradio と、グラフの日本語表示用のパッケージが入ります（1分ほど）。
途中で「ランタイムを再起動してください」と出た場合は、再起動して**手順1からやり直して**ください。

In [ ]:
%pip install -q -r requirements.txt

## 手順3.（任意）ルールの自動テスト
盤面の数（5,478通り）や勝敗判定、`data/sample.csv` との一致などを確認します。最後に `passed` と出ればOKです。

In [ ]:
%pip install -q pytest
!python -m pytest -q

## 手順4. Gradioなしで実験してみる（動作確認）
画面を使わずに、コードから直接学習してグラフを出します。まずここで動くことを確認すると、問題が起きたときに原因を切り分けやすくなります（30秒ほどかかります）。
数値を変えて再実行すれば、条件による正答率の違いを確かめられます。

In [ ]:
from IPython.display import display
from tictactoe_nn.data import build_dataset
from tictactoe_nn.plots import plot_confusion, plot_history
from tictactoe_nn.train import TrainConfig, describe_model, majority_baseline, run_multiple

cfg = TrainConfig(
    board_set="reachable",   # "reachable"=実際に現れる盤面のみ / "all"=全パターン
    input_repr="onehot",     # "onehot"=27個 / "numeric"=数値9個
    n_train=3000,            # 学習データ数（学習に使える上限を超えると自動で調整）
    epochs=30,               # 学習回数
    hidden_layers=2,         # 隠れ層の数（0で隠れ層なし）
    units=64,                # 隠れ層のユニット数
    activation="relu",       # "relu" / "tanh" / "sigmoid" / "leaky_relu"
    lr=1e-3,                 # 学習率
    n_runs=3,                # 同じ条件での実行回数（平均を取る）
)
dataset = build_dataset(cfg.board_set)
multi = run_multiple(cfg, dataset)

mean, std = multi.final("test_acc")
print(describe_model(cfg))
print(f"テストデータの正答率: {mean * 100:.1f}% ± {std * 100:.1f}")
print(f"参考: 常に最多クラスと答えるだけの正答率: {majority_baseline(dataset) * 100:.1f}%")
display(plot_history(multi, majority_baseline(dataset)))
display(plot_confusion(multi))

## 手順5. Web画面（Gradio）を起動
出てきた画面で、左で条件を設定し、「① 学習実験」→「② 学習データ数と正答率」→「③ 盤面を判定」を試せます。
終了するときは、このセルの実行を停止（■ボタン）してください。

**公開リンクについて:** `SHARE = None` のとき、Colab では Gradio が自動で公開リンク（`https://….gradio.live`）を作ることがあります。
リンクを知っている人は誰でも画面を開けるため、他の人に送らないでください（使い終わって停止すれば無効になります）。
`SHARE = False` にすると公開リンクを作りませんが、Colab では画面が表示されないことがあります。その場合は `None` に戻してください。

In [ ]:
# ▼▼▼ ★編集してよい場所 ▼▼▼
SHARE = None   # None=Gradioにおまかせ / False=公開リンクを作らない / True=必ず公開リンクを作る
# ▲▲▲ ここまで ▲▲▲

from tictactoe_nn.app import launch

launch(share=SHARE, debug=True)